# SumCheck Memory-Bound Optimizations

> **ECE-9413 Assignment 2 — Memory Optimization Deep Dive**
>
> SumCheck is **memory-bandwidth bound**, not compute bound. This notebook systematically attacks every source of memory inefficiency:
>
> | # | Optimization | Expected Gain |
> |---|---|---|
> | 1 | Roofline analysis & arithmetic intensity baseline | (diagnostic) |
> | 2 | Kernel fusion via `jax.jit` | 2–3× fewer HBM reads |
> | 3 | Avoid gather: reshape instead of stride-2 index | eliminate copy |
> | 4 | `vmap` degree loop → batch g(t) computation | eliminate d kernel launches |
> | 5 | Fused single-pass claim0 (sum while computing expression) | 1 kernel instead of 2 |
> | 6 | `lax.scan` outer round loop → XLA-native sequential | eliminate n Python roundtrips |
> | 7 | In-place streaming fold (no intermediate table copy) | halve working set per round |
> | 8 | Structure-of-arrays vs Array-of-structures layout | coalesced access patterns |
> | 9 | Chunked streaming for n≥24 (data > L2 cache) | sustain peak HBM BW |
> | 10 | Combined: fully fused JIT-compiled sumcheck | all gains stacked |

---

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import jax
import jax.numpy as jnp
from jax import lax
import numpy as np
import time
import functools

jax.config.update("jax_enable_x64", True)

print("JAX version :", jax.__version__)
print("Backend     :", jax.default_backend())
print("Devices     :", jax.devices())

# Canonical primes
Q32 = jnp.uint32(4_294_967_291)   # 2^32 - 5, largest 32-bit prime
Q_small = jnp.uint32(17)           # for readable worked examples

---
## Section 0 — Why SumCheck Is Memory-Bandwidth Bound

### Arithmetic Intensity (AI)

$$\text{AI} = \frac{\text{FLOPs}}{\text{bytes moved}}$$

A kernel is **compute-bound** if AI exceeds the hardware's ridge point; otherwise it is **memory-bound**.

For `mod_mul_32(a, b, q)` on N elements:
- **FLOPs**: 2 ops per element (one multiply + one mod)
- **Bytes moved**: 3 × N × 4 bytes (read a, read b, write result)
- **AI** = 2N / (12N) = **0.17 FLOPs/byte**

A100 ridge point ≈ **200 FLOPs/byte** (for float32). SumCheck's AI is **1000× below the ridge** — solidly in the memory-bound regime.

This means: **no amount of clever arithmetic can help**. The only lever is reducing bytes moved.

In [ ]:
# ── Roofline analysis ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Hardware specs (A100 / T4 for Colab)
hbm_bw_GBs   = 600    # T4: ~300 GB/s,  A100: ~2000 GB/s — use conservative T4
peak_TFLOPS   = 65     # T4 int8 peak; fp32 ≈ 8 TFLOPS — rough
ridge_point   = peak_TFLOPS * 1e12 / (hbm_bw_GBs * 1e9)  # FLOPs/byte

ops = {
    "mod_mul_32 (unfused)": (2, 12),       # FLOPs, bytes per element
    "mod_add_32 (unfused)": (2, 12),
    "MLE update (3 ops, unfused)": (6, 36), # 3 kernels, 9 passes
    "MLE update (fused)": (6, 12),          # 1 kernel, 3 passes
    "claim0 reduction": (1, 4),
    "Matrix mult (reference)": (2000, 8),   # high AI
}

fig, ax = plt.subplots(figsize=(10, 5))
AI_range = np.logspace(-3, 4, 300)
roofline = np.minimum(peak_TFLOPS * 1e12, AI_range * hbm_bw_GBs * 1e9) / 1e9
ax.loglog(AI_range, roofline, 'k-', lw=2, label='Roofline (T4)')

colors = plt.cm.tab10(np.linspace(0, 1, len(ops)))
for (name, (flops, bytes_)), color in zip(ops.items(), colors):
    ai = flops / bytes_
    perf = min(peak_TFLOPS * 1e12, ai * hbm_bw_GBs * 1e9) / 1e9
    ax.scatter(ai, perf, s=120, color=color, zorder=5)
    ax.annotate(name, (ai, perf), textcoords='offset points',
                xytext=(5, 5), fontsize=8, color=color)

ax.axvline(ridge_point, color='gray', linestyle='--', alpha=0.5)
ax.text(ridge_point * 1.1, 1, f'Ridge {ridge_point:.0f} FLOPs/B', fontsize=8, color='gray')
ax.set_xlabel('Arithmetic Intensity (FLOPs / byte)')
ax.set_ylabel('Performance (GFLOPs/s)')
ax.set_title('Roofline Model: SumCheck operations are deep in memory-bound regime')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nRidge point: {ridge_point:.1f} FLOPs/byte")
print(f"mod_mul_32 AI: {2/12:.3f} FLOPs/byte   ({ridge_point/(2/12):.0f}× below ridge)")
print(f"Fused MLE AI:  {6/12:.3f} FLOPs/byte   ({ridge_point/(6/12):.0f}× below ridge)")
print("\n→ SumCheck is ALWAYS memory-bound. Reduce bytes moved, not FLOPs.")

### What Moves Bytes?

Every HBM access in SumCheck:

```
Operation               Reads        Writes       Total (×N×4 bytes)
──────────────────────────────────────────────────────────────────────
mod_mul_32(a, b, q)      2 arrays     1 array      3×
mod_add_32(a, b, q)      2 arrays     1 array      3×
mod_sub_32(a, b, q)      2 arrays     1 array      3×
table[0::2]  (gather)    1 array      1 array      2×  (stride-2)
MLE update (3 unfused)   6 arrays     3 arrays     9×
MLE update (1 fused)     2 arrays     1 array      3×  ← 3× saving
jnp.sum()                1 array      1 scalar     1×
```

Every optimization below reduces one or more of these.

---
## Section 1 — Baseline (Unfused, Pythonic)

The reference implementation runs each operation as a separate kernel. We use this as the baseline to measure against.

In [ ]:
# ── Baseline primitives (no JIT, no fusion) ────────────────────────────────

def mod_add_32(a, b, q):
    return ((a.astype(jnp.uint64) + b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

def mod_sub_32(a, b, q):
    q64 = jnp.uint64(q)
    return ((a.astype(jnp.uint64) + q64 - b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def mod_mul_32(a, b, q):
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

def mle_update_32(zero_eval, one_eval, target_eval, *, q):
    # 3 separate kernels — unfused
    diff   = mod_sub_32(one_eval, zero_eval, q)     # kernel 1
    scaled = mod_mul_32(diff, target_eval, q)        # kernel 2
    return mod_add_32(scaled, zero_eval, q)          # kernel 3

def eval_expression(tables, expression, q):
    total = None
    for term in expression:
        term_val = tables[term[0]]
        for var in term[1:]:
            term_val = mod_mul_32(term_val, tables[var], q)
        total = term_val if total is None else mod_add_32(total, term_val, q)
    return total

def sumcheck_32_baseline(eval_tables, *, q, expression, challenges, num_rounds):
    """Baseline: correct but unoptimized — every op is a separate kernel."""
    degree = sum(len(term) for term in expression)
    n_eval = degree + 1

    # claim0: sum over all 2^n Boolean points
    f_vals = eval_expression(eval_tables, expression, q)
    claim0 = (jnp.sum(f_vals.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

    tables = dict(eval_tables)
    all_round_evals = []

    for round_idx in range(num_rounds):
        # Step 1: split — stride-2 gather (two memory copies)
        evens = {var: tbl[0::2] for var, tbl in tables.items()}
        odds  = {var: tbl[1::2] for var, tbl in tables.items()}

        # Step 2: evaluate g(t) for t = 0..degree
        round_evals = []
        for t_int in range(n_eval):
            t = jnp.array(t_int, dtype=jnp.uint32)
            if t_int == 0:
                tables_t = evens
            elif t_int == 1:
                tables_t = odds
            else:
                tables_t = {var: mle_update_32(evens[var], odds[var], t, q=q)
                            for var in tables}
            vals = eval_expression(tables_t, expression, q)
            g_t  = (jnp.sum(vals.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)
            round_evals.append(g_t)

        all_round_evals.append(jnp.stack(round_evals))

        # Step 3: fold with challenge
        if round_idx < len(challenges):
            r = challenges[round_idx]
            tables = {var: mle_update_32(evens[var], odds[var], r, q=q)
                      for var in tables}

    return claim0, jnp.stack(all_round_evals)


# Quick correctness check on tiny example
tables_tiny = {'a': jnp.array([0,1,2,3,4,5,6,7], dtype=jnp.uint32)}
chals_tiny  = jnp.array([8, 5], dtype=jnp.uint32)
c0, re = sumcheck_32_baseline(tables_tiny, q=Q_small, expression=[['a']],
                               challenges=chals_tiny, num_rounds=3)
assert int(c0) == 11, f"claim0={c0}"
assert re.tolist() == [[12, 16], [3, 7], [1, 5]], f"round_evals={re}"
print("Baseline correctness: PASS")
print(f"claim0={int(c0)}, round_evals={re.tolist()}")

In [ ]:
# ── Benchmark harness ─────────────────────────────────────────────────────
def bench_sumcheck(fn, tables, *, q, expression, challenges, num_rounds,
                   warmup=3, runs=10, label=""):
    for _ in range(warmup):
        c0, re = fn(tables, q=q, expression=expression,
                    challenges=challenges, num_rounds=num_rounds)
        re.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(runs):
        c0, re = fn(tables, q=q, expression=expression,
                    challenges=challenges, num_rounds=num_rounds)
        re.block_until_ready()
    ms = (time.perf_counter() - t0) / runs * 1000
    print(f"  {label:40s}  {ms:8.2f} ms")
    return ms, c0, re

# Setup benchmark tables
N_BENCH = 20          # num_rounds
N       = 2**N_BENCH  # 1M elements
key     = jax.random.PRNGKey(42)
EXPR    = [['a', 'b', 'c']]  # degree=3
VARS    = ['a', 'b', 'c']

tables_bench = {v: jax.random.randint(key, (N,), 0, int(Q32), dtype=jnp.uint32)
                for v in VARS}
chals_bench  = jax.random.randint(key, (N_BENCH-1,), 1, int(Q32), dtype=jnp.uint32)

print(f"Benchmark config: n={N_BENCH}, N=2^{N_BENCH}={N:,}, expr=a*b*c")
print(f"Table size per var: {N*4/1024**2:.1f} MB   Total: {N*4*3/1024**2:.1f} MB")
print()

results = {}
ms, c0_ref, re_ref = bench_sumcheck(
    sumcheck_32_baseline, tables_bench,
    q=Q32, expression=EXPR, challenges=chals_bench, num_rounds=N_BENCH,
    label="[0] Baseline (unfused)"
)
results['baseline'] = ms

---
## Optimization 1 — Kernel Fusion via `jax.jit`

### The problem

Without JIT, `mle_update_32` launches **3 separate CUDA kernels**:
```
kernel 1: diff   = odds - evens   → write N/2 elements to HBM
kernel 2: scaled = diff * r        → read N/2 from HBM, write N/2 to HBM
kernel 3: result = scaled + evens  → read N/2 from HBM, write N/2 to HBM
```
**HBM traffic: 9 × (N/2) × 4 bytes = 18N bytes**

### With JIT
XLA fuses all three into one kernel that keeps `diff` and `scaled` in registers:
```
kernel 1 (fused): read odds, read evens → compute → write result
```
**HBM traffic: 3 × (N/2) × 4 bytes = 6N bytes — 3× less**

In [ ]:
# ── Optimization 1: JIT the inner ops ─────────────────────────────────────

@jax.jit
def mle_update_32_fused(zero_eval, one_eval, target_eval, q):
    """Fused: XLA keeps diff and scaled in registers — 1 kernel instead of 3."""
    q64    = jnp.uint64(q)
    diff   = (one_eval.astype(jnp.uint64) + q64 - zero_eval.astype(jnp.uint64)) % q64
    scaled = (diff * target_eval.astype(jnp.uint64)) % q64
    result = (scaled + zero_eval.astype(jnp.uint64)) % q64
    return result.astype(jnp.uint32)


# Demonstrate fusion savings with roofline numbers
N_demo = 2**20
a_demo = jax.random.randint(key, (N_demo,), 0, int(Q32), dtype=jnp.uint32)
b_demo = jax.random.randint(key, (N_demo,), 0, int(Q32), dtype=jnp.uint32)
r_demo = jnp.uint32(12345678)

# Unfused (separate JIT for each op)
jit_sub = jax.jit(lambda a,b,q: mod_sub_32(a,b,q))
jit_mul = jax.jit(lambda a,b,q: mod_mul_32(a,b,q))
jit_add = jax.jit(lambda a,b,q: mod_add_32(a,b,q))

def unfused_mle(z, o, r, q):
    diff   = jit_sub(o, z, q)
    scaled = jit_mul(diff, r, q)
    return jit_add(scaled, z, q)

# Warm up
_ = unfused_mle(a_demo, b_demo, r_demo, Q32).block_until_ready()
_ = mle_update_32_fused(a_demo, b_demo, r_demo, Q32).block_until_ready()

RUNS = 50
t0 = time.perf_counter()
for _ in range(RUNS):
    unfused_mle(a_demo, b_demo, r_demo, Q32).block_until_ready()
ms_unfused = (time.perf_counter() - t0) / RUNS * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    mle_update_32_fused(a_demo, b_demo, r_demo, Q32).block_until_ready()
ms_fused = (time.perf_counter() - t0) / RUNS * 1000

bytes_unfused = 9 * N_demo * 4
bytes_fused   = 3 * N_demo * 4
bw_unfused_GBs = bytes_unfused / (ms_unfused / 1000) / 1e9
bw_fused_GBs   = bytes_fused   / (ms_fused   / 1000) / 1e9

print("MLE Update Fusion Comparison (N=2^20)")
print("=" * 55)
print(f"  Unfused (3 kernels):  {ms_unfused:.3f} ms   HBM BW used: {bw_unfused_GBs:.1f} GB/s")
print(f"  Fused   (1 kernel):   {ms_fused:.3f} ms   HBM BW used: {bw_fused_GBs:.1f} GB/s")
print(f"  Speedup from fusion:  {ms_unfused/ms_fused:.2f}×")
print(f"  HBM traffic ratio:    {bytes_unfused/bytes_fused:.1f}× (9 passes → 3 passes)")

---
## Optimization 2 — Reshape Instead of Stride-2 Gather

### The problem

`table[0::2]` does a **stride-2 gather**: reads every other element. On GPU, this causes **non-coalesced memory access** — the hardware reads a full 128-byte cache line but only uses half the bytes.

### The fix

Interpret the flat `[2^k]` table as shape `[2^(k-1), 2]` and slice along axis 1:

```python
# Stride-2 gather (bad):        reshape + slice (good):
evens = table[0::2]             reshaped = table.reshape(-1, 2)
odds  = table[1::2]             evens = reshaped[:, 0]   # contiguous
                                odds  = reshaped[:, 1]   # contiguous
```

After reshape, `evens` and `odds` are contiguous in memory — **fully coalesced GPU access**.

Alternatively: keep the table in `[2, N/2]` shape from the start so splits are free slices.

In [ ]:
# ── Optimization 2: Reshape vs stride-2 gather ────────────────────────────

N_test = 2**22   # 4M elements to make memory access pattern visible
t_flat = jax.random.randint(key, (N_test,), 0, int(Q32), dtype=jnp.uint32)

@jax.jit
def split_stride(table):
    return table[0::2], table[1::2]   # stride-2 gather

@jax.jit
def split_reshape(table):
    r = table.reshape(-1, 2)          # view as (N/2, 2)
    return r[:, 0], r[:, 1]           # contiguous slices

# Warm up
_ = split_stride(t_flat)[0].block_until_ready()
_ = split_reshape(t_flat)[0].block_until_ready()

RUNS = 100
t0 = time.perf_counter()
for _ in range(RUNS):
    e, o = split_stride(t_flat)
    e.block_until_ready()
ms_stride = (time.perf_counter() - t0) / RUNS * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    e, o = split_reshape(t_flat)
    e.block_until_ready()
ms_reshape = (time.perf_counter() - t0) / RUNS * 1000

bytes_moved = 3 * N_test * 4   # read input, write 2 outputs
bw_stride  = bytes_moved / (ms_stride  / 1000) / 1e9
bw_reshape = bytes_moved / (ms_reshape / 1000) / 1e9

print("Split Strategy Comparison (N=2^22)")
print("=" * 55)
print(f"  Stride-2 gather:   {ms_stride:.3f} ms   eff. BW: {bw_stride:.1f} GB/s")
print(f"  Reshape + slice:   {ms_reshape:.3f} ms   eff. BW: {bw_reshape:.1f} GB/s")
print(f"  Speedup:           {ms_stride/ms_reshape:.2f}×")
print()
print("Key insight: reshape is O(0) cost — it's just a view, no data movement.")
print("The slice along axis-1 accesses contiguous cache lines → coalesced.")

# Verify correctness
e_s, o_s = split_stride(t_flat)
e_r, o_r = split_reshape(t_flat)
assert jnp.array_equal(e_s, e_r) and jnp.array_equal(o_s, o_r), "Results differ!"
print("Correctness: PASS")

---
## Optimization 3 — `vmap` the Degree Loop

### The problem

The inner loop `for t in range(degree+1)` runs `d+1` iterations in Python. For `d=5`, that's 6 separate kernel launches with Python overhead between them.

### The fix

Use `jax.vmap` to evaluate all `g(0), g(1), ..., g(d)` **simultaneously** in a single batched kernel:

```python
t_vals = jnp.arange(num_eval_points, dtype=jnp.uint32)   # shape (d+1,)
# vmap over t_vals → compute g(t) for all t in one kernel
all_g_t = vmap(compute_g)(t_vals)   # shape (d+1,)
```

This eliminates `d` Python-level synchronization points and gives XLA full visibility to schedule the batch optimally.

In [ ]:
# ── Optimization 3: vmap over t values ────────────────────────────────────

def make_g_evaluator(evens_dict, odds_dict, expression, q):
    """Return a function (t: scalar uint32) -> g(t) scalar uint32."""
    vars_ = list(evens_dict.keys())
    evens_stack = jnp.stack([evens_dict[v] for v in vars_])  # (V, N/2)
    odds_stack  = jnp.stack([odds_dict[v]  for v in vars_])  # (V, N/2)
    var_idx     = {v: i for i, v in enumerate(vars_)}

    def g_at_t(t):
        """Compute g(t) for a single uint32 scalar t."""
        q64 = jnp.uint64(q)
        t64 = t.astype(jnp.uint64)

        # MLE update for each variable: zeros + t*(ones-zeros) mod q
        diff   = (odds_stack.astype(jnp.uint64) + q64 - evens_stack.astype(jnp.uint64)) % q64
        scaled = (diff * t64) % q64
        tables_t = (scaled + evens_stack.astype(jnp.uint64)) % q64  # (V, N/2)

        # Evaluate expression
        total = jnp.zeros(evens_stack.shape[1], dtype=jnp.uint64)
        for term in expression:
            term_val = tables_t[var_idx[term[0]]]
            for var in term[1:]:
                term_val = (term_val * tables_t[var_idx[var]]) % q64
            total = (total + term_val) % q64

        return (jnp.sum(total) % q64).astype(jnp.uint32)

    return g_at_t


# Build test case
N_test = 2**16
EXPR_test = [['a', 'b', 'c']]
degree = 3
n_eval = degree + 1
tables_t = {v: jax.random.randint(key, (N_test,), 0, int(Q32), dtype=jnp.uint32)
            for v in ['a', 'b', 'c']}
reshaped = {v: tables_t[v].reshape(-1, 2) for v in tables_t}
evens = {v: reshaped[v][:, 0] for v in reshaped}
odds  = {v: reshaped[v][:, 1] for v in reshaped}

g_fn = make_g_evaluator(evens, odds, EXPR_test, Q32)

# Sequential loop approach
@functools.partial(jax.jit, static_argnums=())
def compute_round_evals_loop(evens_stack, odds_stack, q, n_eval, degree):
    # Will be traced loop — not great
    pass

# vmap approach
g_batched = jax.jit(jax.vmap(g_fn))
t_vals    = jnp.arange(n_eval, dtype=jnp.uint32)

# Warm up
_ = g_batched(t_vals).block_until_ready()

# Benchmark loop vs vmap
RUNS = 50

# Loop (each call JIT-compiled but separate kernel launches)
g_single = jax.jit(g_fn)
_ = g_single(jnp.uint32(0)).block_until_ready()

t0 = time.perf_counter()
for _ in range(RUNS):
    results_loop = jnp.stack([g_single(jnp.uint32(t)) for t in range(n_eval)])
    results_loop.block_until_ready()
ms_loop = (time.perf_counter() - t0) / RUNS * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    results_vmap = g_batched(t_vals)
    results_vmap.block_until_ready()
ms_vmap = (time.perf_counter() - t0) / RUNS * 1000

print("Degree Loop: Python loop vs vmap (N=2^16, degree=3, n_eval=4)")
print("=" * 55)
print(f"  Python loop (4 kernel launches): {ms_loop:.3f} ms")
print(f"  vmap (1 batched kernel):          {ms_vmap:.3f} ms")
print(f"  Speedup: {ms_loop/ms_vmap:.2f}×")
print()
assert jnp.array_equal(results_loop, results_vmap), "Results differ!"
print("Correctness: PASS")

---
## Optimization 4 — `lax.scan` for the Outer Round Loop

### The problem

The outer `for round_idx in range(num_rounds)` loop runs in Python. Each iteration:
1. Dispatches JAX ops (Python overhead: ~10–100 µs)
2. Reads the challenge from a Python list
3. Waits for the previous round's table before launching the next

For `n=20` rounds, that's up to **2 ms of pure Python overhead**.

### The fix

`lax.scan` compiles the loop body **once** and executes all iterations inside a single XLA program. Python is bypassed entirely after the first JIT trace.

```python
def round_body(tables_carry, challenge):
    # compute g(t), fold → return (new_tables, round_evals)
    ...
    return new_tables, round_evals_row

final_tables, all_round_evals = lax.scan(round_body, init_tables, challenges)
```

**Trade-off**: `lax.scan` requires static shapes — the challenge array must be pre-allocated. But for SumCheck that's already the case.

In [ ]:
# ── Optimization 4: lax.scan outer round loop ─────────────────────────────

def sumcheck_32_scan(eval_tables, *, q, expression, challenges, num_rounds):
    """
    SumCheck with lax.scan — the round loop runs entirely inside XLA,
    eliminating Python dispatch overhead for all n rounds.
    """
    degree  = sum(len(term) for term in expression)
    n_eval  = degree + 1
    vars_   = list(eval_tables.keys())
    V       = len(vars_)
    q64     = jnp.uint64(q)

    # Stack all variable tables: shape (V, N)
    tables_stacked = jnp.stack([eval_tables[v] for v in vars_])  # (V, N)

    # Build term indices for expression evaluation
    # term_indices[term_i] = list of variable indices in that term
    var_idx = {v: i for i, v in enumerate(vars_)}

    def eval_expr_stacked(tables_V_N):
        """Evaluate expression on stacked tables of shape (V, N)."""
        total = jnp.zeros(tables_V_N.shape[1], dtype=jnp.uint64)
        for term in expression:
            term_val = tables_V_N[var_idx[term[0]]].astype(jnp.uint64)
            for var in term[1:]:
                term_val = (term_val * tables_V_N[var_idx[var]].astype(jnp.uint64)) % q64
            total = (total + term_val) % q64
        return total

    # claim0 — sum over all 2^n points
    f_vals = eval_expr_stacked(tables_stacked)
    claim0 = (jnp.sum(f_vals) % q64).astype(jnp.uint32)

    # Pad challenges to full num_rounds length (last element unused by prover)
    # lax.scan needs equal-length arrays
    r_padded = jnp.concatenate([
        challenges,
        jnp.zeros(num_rounds - len(challenges), dtype=jnp.uint32)
    ])

    t_vals = jnp.arange(n_eval, dtype=jnp.uint32)  # [0, 1, ..., degree]

    def round_body(tables_carry, r):
        """One sumcheck round. tables_carry: (V, N_k)"""
        N_k  = tables_carry.shape[1]
        half = N_k // 2

        # Split via reshape — no stride-2 gather
        reshaped = tables_carry.reshape(V, half, 2)   # (V, N/2, 2)
        evens    = reshaped[:, :, 0]                   # (V, N/2)
        odds     = reshaped[:, :, 1]                   # (V, N/2)

        # vmap over t: compute g(t) for all eval points simultaneously
        def g_at_t(t):
            t64    = t.astype(jnp.uint64)
            diff   = (odds.astype(jnp.uint64) + q64 - evens.astype(jnp.uint64)) % q64
            tables_t = (diff * t64 % q64 + evens.astype(jnp.uint64)) % q64  # (V, N/2)
            vals   = eval_expr_stacked(tables_t.astype(jnp.uint32))
            return (jnp.sum(vals.astype(jnp.uint64)) % q64).astype(jnp.uint32)

        round_evals_row = jax.vmap(g_at_t)(t_vals)  # (n_eval,)

        # MLE fold with challenge r
        r64      = r.astype(jnp.uint64)
        diff     = (odds.astype(jnp.uint64) + q64 - evens.astype(jnp.uint64)) % q64
        new_tabs = (diff * r64 % q64 + evens.astype(jnp.uint64)) % q64  # (V, N/2)
        new_tabs = new_tabs.astype(jnp.uint32)

        return new_tabs, round_evals_row

    # Run all rounds inside XLA — no Python loop overhead
    _, all_round_evals = lax.scan(round_body, tables_stacked, r_padded)

    return claim0, all_round_evals


# JIT-compile once
sumcheck_32_scan_jit = jax.jit(
    sumcheck_32_scan,
    static_argnames=['expression', 'num_rounds']
)

# Correctness check
c0_s, re_s = sumcheck_32_scan_jit(
    tables_tiny, q=Q_small, expression=tuple(map(tuple, [['a']])),
    challenges=chals_tiny, num_rounds=3
)
print("lax.scan correctness:", "PASS" if int(c0_s) == 11 else f"FAIL (got {int(c0_s)})")
print(f"  claim0={int(c0_s)}, round_evals={re_s.tolist()}")

In [ ]:
# Benchmark: baseline vs scan
print(f"Benchmarking lax.scan version (n={N_BENCH}, N=2^{N_BENCH})")
print()

ms_scan, c0_scan, re_scan = bench_sumcheck(
    lambda tbls, **kw: sumcheck_32_scan_jit(tbls, **kw),
    tables_bench, q=Q32,
    expression=tuple(map(tuple, EXPR)),
    challenges=chals_bench, num_rounds=N_BENCH,
    label="[4] lax.scan + vmap + reshape"
)
results['scan'] = ms_scan

print()
print(f"  Baseline:          {results['baseline']:.2f} ms")
print(f"  lax.scan:          {ms_scan:.2f} ms")
print(f"  Speedup:           {results['baseline']/ms_scan:.2f}×")

---
## Optimization 5 — Single-Pass Claim0 (Fused Sum + Expression)

### The problem

Computing `claim0` currently does:
1. Evaluate expression → write `f_vals` array to HBM (N × 4 bytes)
2. Sum `f_vals` → read `f_vals` back from HBM (N × 4 bytes)

That's 2× N reads + 1× N write = **3N memory passes** for the sum alone.

### The fix

Fuse the expression evaluation and the sum into a **single JIT-compiled kernel** that accumulates into a scalar without ever materializing the full `f_vals` array in HBM:

```python
@jax.jit
def fused_sum_expression(tables_stacked, q):
    f_vals = eval_expr_stacked(tables_stacked)  # computed in registers/shared mem
    return (jnp.sum(f_vals) % uint64(q)).astype(uint32)
```

With JIT, XLA sees `eval → sum` as one graph and fuses them. The intermediate `f_vals` stays in register file / shared memory — **never written to HBM**.

In [ ]:
# ── Optimization 5: fused claim0 ─────────────────────────────────────────

N_test = 2**20
tbls   = {v: jax.random.randint(key, (N_test,), 0, int(Q32), dtype=jnp.uint32)
          for v in ['a', 'b', 'c']}
vars_  = ['a', 'b', 'c']
stacked = jnp.stack([tbls[v] for v in vars_])
var_idx = {v: i for i, v in enumerate(vars_)}
expr    = [['a', 'b', 'c']]

def eval_expr_raw(tables):
    total = None
    for term in expr:
        tv = tables[term[0]]
        for var in term[1:]:
            tv = mod_mul_32(tv, tables[var], Q32)
        total = tv if total is None else mod_add_32(total, tv, Q32)
    return total

# Unfused: eval then sum (two separate JIT calls)
jit_eval = jax.jit(lambda t: eval_expr_raw(t))
jit_sum  = jax.jit(lambda f: (jnp.sum(f.astype(jnp.uint64)) % jnp.uint64(Q32)).astype(jnp.uint32))

def claim0_unfused(tables):
    f = jit_eval(tables)
    return jit_sum(f)

# Fused: one JIT over the whole graph
@jax.jit
def claim0_fused(tables_stack):
    q64 = jnp.uint64(Q32)
    total = jnp.zeros(tables_stack.shape[1], dtype=jnp.uint64)
    for term in expr:
        tv = tables_stack[var_idx[term[0]]].astype(jnp.uint64)
        for var in term[1:]:
            tv = (tv * tables_stack[var_idx[var]].astype(jnp.uint64)) % q64
        total = (total + tv) % q64
    return (jnp.sum(total) % q64).astype(jnp.uint32)  # fused reduce

# Warm up
_ = claim0_unfused(tbls).block_until_ready()
_ = claim0_fused(stacked).block_until_ready()

RUNS = 50
t0 = time.perf_counter()
for _ in range(RUNS):
    claim0_unfused(tbls).block_until_ready()
ms_unfused_c0 = (time.perf_counter() - t0) / RUNS * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    claim0_fused(stacked).block_until_ready()
ms_fused_c0 = (time.perf_counter() - t0) / RUNS * 1000

# HBM traffic analysis
# Unfused: read 3 tables (input) + write f_vals + read f_vals = 5N reads+writes
# Fused:   read 3 tables, accumulate in shared mem, write scalar = 3N reads + 0 write
bytes_unfused_c0 = (3 + 1 + 1) * N_test * 4  # read 3 inputs, write f, read f
bytes_fused_c0   = 3            * N_test * 4  # read 3 inputs only

print("Claim0 Fusion Comparison (N=2^20, expr=a*b*c)")
print("=" * 55)
print(f"  Unfused (eval→sum): {ms_unfused_c0:.3f} ms")
print(f"  Fused   (one pass): {ms_fused_c0:.3f} ms")
print(f"  Speedup:            {ms_unfused_c0/ms_fused_c0:.2f}×")
print(f"  HBM traffic: {bytes_unfused_c0/1e6:.0f} MB → {bytes_fused_c0/1e6:.0f} MB")
print(f"  ({bytes_unfused_c0/bytes_fused_c0:.1f}× less data moved)")

---
## Optimization 6 — In-Place / Streaming MLE Fold

### The problem

After each round, the current code creates a **new table** of size N/2 while keeping the old table of size N alive until Python GC collects it. Peak memory usage = **1.5× N** elements.

For `n=24` with 6 variables: `6 × 2^24 × 4 bytes = 384 MB` per round — enough to exhaust GPU L2 cache.

### The fix

Use `lax.scan` with a **halving carry**: at each step the carry is the folded (smaller) table. JAX/XLA can reuse the buffer because the new size is exactly half the old.

Also: compute the fold and the next round's g(t) in the **same pass** over the data, avoiding a second read of the folded table.

In [ ]:
# ── Optimization 6: Memory footprint during scan ───────────────────────────
# Demonstrate that lax.scan naturally achieves the streaming fold
# because XLA owns the buffer management and never needs both the
# N-element and N/2-element arrays at the same time.

def memory_footprint_analysis():
    print("Memory footprint analysis across rounds")
    print("=" * 60)
    print(f"{'n':3s}  {'N=2^n':10s}  {'V vars':6s}  {'Peak (Python)':15s}  {'Peak (scan)':12s}")
    print("-" * 60)

    for n in [16, 20, 22, 24]:
        for V in [3, 6]:
            N = 2**n
            # Python loop: at round transition, both N and N/2 tables exist simultaneously
            peak_python_bytes = V * N * 4 + V * (N // 2) * 4   # old + new
            # lax.scan: XLA reuses buffers, only ever needs current table
            peak_scan_bytes   = V * N * 4                        # only initial (then shrinks)
            # In practice with lax.scan the largest single buffer is N, N/2 at next step
            # XLA will double-buffer but reuse the same slot — peak stays at ~V×N×4

            print(f"  {n:2d}  {N:10,}  {V:6d}  "
                  f"{peak_python_bytes/1024**2:8.1f} MB        "
                  f"{peak_scan_bytes/1024**2:8.1f} MB")

memory_footprint_analysis()

print()
print("With lax.scan: XLA's buffer allocator sees the full computation graph")
print("and can reuse the N-element buffer storage once the N/2 table is written.")
print("The Python loop cannot do this — both arrays are Python-visible at once.")

---
## Optimization 7 — Structure of Arrays (SoA) Layout

### The problem

The expression `a*b*c` needs elements `a[i]`, `b[i]`, `c[i]` for each i. If tables are stored separately (as a Python dict), reading all three for index `i` requires 3 different memory addresses — potentially 3 different cache lines.

### Option A: Array of Structures (AoS)
```
interleaved: [a0, b0, c0, a1, b1, c1, a2, b2, c2, ...]
```
Reading `a[i], b[i], c[i]` → 1 cache line. But then striding for vectorization is bad.

### Option B: Structure of Arrays (SoA) — stacked
```
shape (V, N): row 0 = all a values, row 1 = all b values, ...
```
Reading the i-th column across all rows → coalesced if we process all variables for one index together. GPU prefers this for vectorized kernels.

### For SumCheck specifically

The stacked `(V, N)` layout (which our `lax.scan` impl already uses) is best because:
- The expression multiply `a*b*c` processes the entire array: all `a[0..N]` then all `b[0..N]` etc.
- Each row access is fully coalesced.
- XLA's `vmap` over variables maps directly to this layout.

In [ ]:
# ── Optimization 7: Layout comparison ─────────────────────────────────────

N_layout = 2**20
V        = 3

# Layout A: dict of separate arrays (baseline)
tbls_dict = {v: jax.random.randint(key, (N_layout,), 0, int(Q32), dtype=jnp.uint32)
             for v in ['a','b','c']}

# Layout B: stacked (V, N) — SoA
tbls_stacked = jnp.stack([tbls_dict['a'], tbls_dict['b'], tbls_dict['c']])  # (3, N)

# Layout C: interleaved (AoS) — shape (N, V)
tbls_interleaved = tbls_stacked.T  # (N, 3) — AoS

q64 = jnp.uint64(Q32)

@jax.jit
def eval_dict(tbls):
    a, b, c = tbls['a'].astype(jnp.uint64), tbls['b'].astype(jnp.uint64), tbls['c'].astype(jnp.uint64)
    return (jnp.sum(a * b % q64 * c % q64) % q64).astype(jnp.uint32)

@jax.jit
def eval_stacked(tbls_VN):
    a, b, c = tbls_VN[0].astype(jnp.uint64), tbls_VN[1].astype(jnp.uint64), tbls_VN[2].astype(jnp.uint64)
    return (jnp.sum(a * b % q64 * c % q64) % q64).astype(jnp.uint32)

@jax.jit
def eval_interleaved(tbls_NV):
    a, b, c = tbls_NV[:, 0].astype(jnp.uint64), tbls_NV[:, 1].astype(jnp.uint64), tbls_NV[:, 2].astype(jnp.uint64)
    return (jnp.sum(a * b % q64 * c % q64) % q64).astype(jnp.uint32)

# Warm up
_ = eval_dict(tbls_dict).block_until_ready()
_ = eval_stacked(tbls_stacked).block_until_ready()
_ = eval_interleaved(tbls_interleaved).block_until_ready()

RUNS = 50
layouts = [
    ("dict (separate arrays)",    lambda: eval_dict(tbls_dict)),
    ("stacked (V,N) SoA",         lambda: eval_stacked(tbls_stacked)),
    ("interleaved (N,V) AoS",     lambda: eval_interleaved(tbls_interleaved)),
]

print(f"Layout comparison for a*b*c sum (N=2^{int(np.log2(N_layout))})")
print("=" * 55)
for name, fn in layouts:
    t0 = time.perf_counter()
    for _ in range(RUNS):
        fn().block_until_ready()
    ms = (time.perf_counter() - t0) / RUNS * 1000
    bw = (3 * N_layout * 4) / (ms / 1000) / 1e9  # read 3 arrays
    print(f"  {name:30s}  {ms:.3f} ms   {bw:.1f} GB/s")

print()
print("Stacked (V,N) is preferred: rows are contiguous, GPU reads each row")
print("with full cache-line utilization.")

---
## Optimization 8 — Chunked Streaming for Large n (n ≥ 22)

### The problem

For `n=24` with 6 variables:
- Table size per variable: `2^24 × 4 = 64 MB`
- Total working set: `6 × 64 MB = 384 MB`
- GPU L2 cache: **40 MB** (A100) / **4 MB** (T4)

The data doesn't fit in L2. Every operation causes L2 evictions → every element is read from HBM multiple times per round.

### The fix: chunked streaming

Divide the table into chunks that **fit in L2** (e.g., 8 MB chunks on A100). Process each chunk through the **full expression** before moving to the next:

```
Without chunking (N=2^24):
  pass 1: read a[0..N], compute a*b partial
  pass 2: read a[0..N] again? No — but b read causes a eviction from L2
  Each intermediate is written to HBM

With chunking (chunk_size = L2/6 vars):
  for chunk in chunks:
      load a[chunk], b[chunk], c[chunk] → all fit in L2
      compute a*b*c → intermediate stays in L2/registers
      accumulate partial sum
  → each element read ONCE from HBM
```

In [ ]:
# ── Optimization 8: Chunked streaming ─────────────────────────────────────

def compute_chunk_size(n_vars, dtype_bytes=4, l2_bytes=None):
    """Compute chunk size that fits all V variable slices in L2."""
    if l2_bytes is None:
        # Detect approximate L2 size
        backend = jax.default_backend()
        if backend == 'gpu':
            # T4 ≈ 4 MB, A100 ≈ 40 MB — be conservative
            l2_bytes = 4 * 1024 * 1024
        else:
            l2_bytes = 8 * 1024 * 1024  # CPU L3
    # Leave 50% for XLA overhead
    usable = l2_bytes // 2
    return usable // (n_vars * dtype_bytes)


@functools.partial(jax.jit, static_argnames=['expression', 'chunk_size'])
def claim0_chunked(tables_stacked, q, expression, chunk_size):
    """
    Compute sum(expression(tables)) mod q using chunk-wise streaming.
    Each chunk fits in L2, ensuring each element is read from HBM exactly once.
    """
    V, N = tables_stacked.shape
    q64  = jnp.uint64(q)
    n_chunks = (N + chunk_size - 1) // chunk_size

    var_names = [chr(ord('a') + i) for i in range(V)]
    var_idx   = {v: i for i, v in enumerate(var_names)}

    def process_chunk(acc, chunk_idx):
        start = chunk_idx * chunk_size
        # Slice chunk for all variables
        chunk = lax.dynamic_slice_in_dim(tables_stacked, start, chunk_size, axis=1)  # (V, chunk_size)

        # Evaluate expression on chunk
        total_chunk = jnp.zeros(chunk_size, dtype=jnp.uint64)
        for term in expression:
            tv = chunk[var_idx[term[0]]].astype(jnp.uint64)
            for var in term[1:]:
                tv = (tv * chunk[var_idx[var]].astype(jnp.uint64)) % q64
            total_chunk = (total_chunk + tv) % q64

        return (acc + jnp.sum(total_chunk)) % q64, None

    chunk_indices = jnp.arange(n_chunks, dtype=jnp.int32)
    total, _ = lax.scan(process_chunk, jnp.uint64(0), chunk_indices)
    return (total % q64).astype(jnp.uint32)


# Test correctness
N_chunk_test = 2**20
V_test = 3
tbls_test = {v: jax.random.randint(key, (N_chunk_test,), 0, int(Q32), dtype=jnp.uint32)
             for v in ['a','b','c']}
stacked_test = jnp.stack([tbls_test[v] for v in ['a','b','c']])
expr_test    = [['a','b','c']]

chunk_sz = compute_chunk_size(V_test)
print(f"Chunk size for {V_test} vars on {jax.default_backend()}: {chunk_sz:,} elements ({chunk_sz*4/1024:.0f} KB/var)")

# Reference claim0
@jax.jit
def claim0_reference(stk, q):
    q64 = jnp.uint64(q)
    a, b, c = stk[0].astype(jnp.uint64), stk[1].astype(jnp.uint64), stk[2].astype(jnp.uint64)
    return (jnp.sum(a * b % q64 * c % q64) % q64).astype(jnp.uint32)

ref  = claim0_reference(stacked_test, Q32)
chkd = claim0_chunked(stacked_test, Q32, tuple(map(tuple, expr_test)), chunk_sz)

print(f"Reference claim0: {int(ref)}")
print(f"Chunked  claim0:  {int(chkd)}")
print(f"Match: {'PASS' if int(ref) == int(chkd) else 'FAIL'}")

# Benchmark: standard vs chunked
_ = ref.block_until_ready()
_ = chkd.block_until_ready()

RUNS = 30
t0 = time.perf_counter()
for _ in range(RUNS):
    claim0_reference(stacked_test, Q32).block_until_ready()
ms_ref = (time.perf_counter() - t0) / RUNS * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    claim0_chunked(stacked_test, Q32, tuple(map(tuple, expr_test)), chunk_sz).block_until_ready()
ms_chunk = (time.perf_counter() - t0) / RUNS * 1000

print(f"\nN=2^20: standard={ms_ref:.2f}ms  chunked={ms_chunk:.2f}ms")
print("(Chunked gains most when data exceeds L2 cache, i.e. n≥22+)")

---
## Optimization 9 — Combined: Fully Fused JIT-Compiled SumCheck

Stacking all optimizations into a single production-quality implementation:

1. **Stacked (V, N) table layout** (SoA, coalesced access)
2. **Reshape split** instead of stride-2 gather
3. **vmap** over degree evaluation points
4. **lax.scan** over rounds (eliminates Python loop overhead)
5. **Single JIT** over the entire computation graph (maximum XLA fusion)
6. **uint64 arithmetic** preserved internally (no intermediate uint32 casts until output)

In [ ]:
# ── Optimization 9: Fully fused sumcheck ──────────────────────────────────

def make_sumcheck_optimized(expression, num_rounds):
    """
    Factory: returns a JIT-compiled sumcheck function for a fixed expression
    and num_rounds. Static args are baked in at trace time → maximum XLA opts.
    """
    degree  = sum(len(term) for term in expression)
    n_eval  = degree + 1
    vars_   = sorted(set(v for term in expression for v in term))
    V       = len(vars_)
    var_idx = {v: i for i, v in enumerate(vars_)}
    t_vals  = jnp.arange(n_eval, dtype=jnp.uint32)

    @jax.jit
    def sumcheck(eval_tables_stacked, q, challenges):
        """
        Args:
          eval_tables_stacked: (V, 2^n) uint32 — rows ordered by vars_
          q:          uint32 scalar
          challenges: (num_rounds-1,) uint32
        Returns:
          claim0:      uint32 scalar
          round_evals: (num_rounds, n_eval) uint32
        """
        q64 = jnp.uint64(q)

        def eval_expr(tbls_VN):
            """Evaluate expression. tbls_VN: (V, K) uint64."""
            total = jnp.zeros(tbls_VN.shape[1], dtype=jnp.uint64)
            for term in expression:
                tv = tbls_VN[var_idx[term[0]]]
                for var in term[1:]:
                    tv = tv * tbls_VN[var_idx[var]] % q64
                total = (total + tv) % q64
            return total

        # claim0: fused eval + sum
        tbls64   = eval_tables_stacked.astype(jnp.uint64)
        f_vals   = eval_expr(tbls64)
        claim0   = (jnp.sum(f_vals) % q64).astype(jnp.uint32)

        # Pad challenges: lax.scan needs fixed-size arrays
        r_padded = jnp.concatenate([
            challenges,
            jnp.zeros(num_rounds - challenges.shape[0], dtype=jnp.uint32)
        ])

        def round_body(carry_VN, r):
            V_, N_k = carry_VN.shape
            half    = N_k // 2

            # Reshape split — coalesced, zero copies
            rsh   = carry_VN.reshape(V_, half, 2)
            evens = rsh[:, :, 0].astype(jnp.uint64)   # (V, N/2) uint64
            odds  = rsh[:, :, 1].astype(jnp.uint64)   # (V, N/2) uint64
            diff  = (odds + q64 - evens) % q64         # (V, N/2) uint64

            # vmap: compute g(t) for all t simultaneously
            def g_at_t(t):
                t64      = t.astype(jnp.uint64)
                tbls_t   = (diff * t64 % q64 + evens) % q64  # (V, N/2)
                vals     = eval_expr(tbls_t)
                return (jnp.sum(vals) % q64).astype(jnp.uint32)

            round_evals_row = jax.vmap(g_at_t)(t_vals)   # (n_eval,) uint32

            # MLE fold with challenge r — fused with diff already computed
            r64     = r.astype(jnp.uint64)
            new_V_N = (diff * r64 % q64 + evens) % q64   # (V, N/2) uint64
            new_carry = new_V_N.astype(jnp.uint32)

            return new_carry, round_evals_row

        _, all_round_evals = lax.scan(
            round_body,
            eval_tables_stacked,
            r_padded
        )
        return claim0, all_round_evals

    return sumcheck, vars_


# Build and test the optimized version
sumcheck_opt, vars_opt = make_sumcheck_optimized(
    expression=[['a', 'b', 'c']],
    num_rounds=3
)

# Test on tiny example (expression=['a'], not ['a','b','c'] — adjust)
sumcheck_opt_a, vars_opt_a = make_sumcheck_optimized(
    expression=[['a']],
    num_rounds=3
)
stacked_tiny = jnp.array([[0,1,2,3,4,5,6,7]], dtype=jnp.uint32)  # (1, 8)
c0_opt, re_opt = sumcheck_opt_a(stacked_tiny, Q_small, chals_tiny)

print("Optimized sumcheck correctness:")
print(f"  claim0={int(c0_opt)}  expected=11  {'PASS' if int(c0_opt)==11 else 'FAIL'}")
print(f"  round_evals={re_opt.tolist()}")
print(f"  expected=[[12, 16], [3, 7], [1, 5]]")

In [ ]:
# ── Benchmark all optimizations side by side ───────────────────────────────

sumcheck_opt_bench, vars_opt_bench = make_sumcheck_optimized(
    expression=[['a','b','c']],
    num_rounds=N_BENCH
)
stacked_bench = jnp.stack([tables_bench[v] for v in vars_opt_bench])

# Warm up
for _ in range(3):
    c0_w, re_w = sumcheck_opt_bench(stacked_bench, Q32, chals_bench)
    re_w.block_until_ready()

RUNS = 10
t0 = time.perf_counter()
for _ in range(RUNS):
    c0_w, re_w = sumcheck_opt_bench(stacked_bench, Q32, chals_bench)
    re_w.block_until_ready()
ms_opt = (time.perf_counter() - t0) / RUNS * 1000
results['optimized'] = ms_opt

print(f"Final Benchmark Summary (n={N_BENCH}, N=2^{N_BENCH}, expr=a*b*c)")
print("=" * 60)
print(f"  [0] Baseline (unfused Python loop):   {results['baseline']:8.2f} ms")
print(f"  [4] lax.scan + vmap + reshape:        {results['scan']:8.2f} ms  ({results['baseline']/results['scan']:.1f}×)")
print(f"  [9] Fully fused (all optimizations):  {ms_opt:8.2f} ms  ({results['baseline']/ms_opt:.1f}×)")
print()

# Effective HBM bandwidth
N = 2**N_BENCH
V = 3
# Per round: read V tables (N/2), compute, fold → write V/2 tables
# Total across all rounds ≈ 2 × V × N × 4 bytes (geometric series)
bytes_total = 2 * V * N * 4
bw_baseline = bytes_total / (results['baseline'] / 1000) / 1e9
bw_opt      = bytes_total / (ms_opt / 1000) / 1e9
print(f"  Effective HBM bandwidth:")
print(f"    Baseline:    {bw_baseline:.1f} GB/s")
print(f"    Optimized:   {bw_opt:.1f} GB/s")

---
## Section 10 — Expression Sweep: All 7 Expressions

The optimized implementation's gain varies by expression because:
- Higher degree → more g(t) evaluation points → vmap helps more
- More variables → larger working set → fusion saves more HBM traffic

In [ ]:
# ── Expression sweep ───────────────────────────────────────────────────────

EXPRESSIONS = (
    [['a']],
    [['a', 'b']],
    [['a', 'b'], ['c']],
    [['a', 'b', 'c']],
    [['a', 'a', 'b', 'b', 'c']],
    [['a', 'b', 'c'], ['d', 'e']],
    [['a', 'b', 'c', 'g'], ['d', 'e', 'g']],
)

N_SWEEP  = 18    # use smaller n for sweep speed
N_sweep  = 2**N_SWEEP
ALL_VARS = ['a', 'b', 'c', 'd', 'e', 'g']

tables_all = {v: jax.random.randint(key, (N_sweep,), 0, int(Q32), dtype=jnp.uint32)
              for v in ALL_VARS}
chals_sw   = jax.random.randint(key, (N_SWEEP-1,), 1, int(Q32), dtype=jnp.uint32)

print(f"Expression sweep: n={N_SWEEP}, N=2^{N_SWEEP}={N_sweep:,}")
print()
print(f"{'Expression':30s}  {'Deg':3s}  {'Vars':4s}  {'Baseline ms':12s}  {'Optimized ms':13s}  {'Speedup'}")
print("-" * 80)

RUNS_SW = 5

for expr in EXPRESSIONS:
    degree   = sum(len(t) for t in expr)
    vars_used = sorted(set(v for term in expr for v in term))
    expr_str  = " + ".join("*".join(t) for t in expr)

    tbls_expr = {v: tables_all[v] for v in vars_used}
    stacked_expr = jnp.stack([tbls_expr[v] for v in vars_used])

    # Baseline
    for _ in range(2):
        sumcheck_32_baseline(tbls_expr, q=Q32, expression=expr,
                             challenges=chals_sw, num_rounds=N_SWEEP).count_nonzero()
    t0 = time.perf_counter()
    for _ in range(RUNS_SW):
        c0b, reb = sumcheck_32_baseline(tbls_expr, q=Q32, expression=expr,
                                        challenges=chals_sw, num_rounds=N_SWEEP)
        reb.block_until_ready()
    ms_base = (time.perf_counter() - t0) / RUNS_SW * 1000

    # Optimized
    sc_opt, vs_opt = make_sumcheck_optimized(expr, N_SWEEP)
    stk = jnp.stack([tbls_expr[v] for v in vs_opt])
    for _ in range(2):
        sc_opt(stk, Q32, chals_sw)[1].block_until_ready()
    t0 = time.perf_counter()
    for _ in range(RUNS_SW):
        c0o, reo = sc_opt(stk, Q32, chals_sw)
        reo.block_until_ready()
    ms_opt_sw = (time.perf_counter() - t0) / RUNS_SW * 1000

    speedup = ms_base / ms_opt_sw
    print(f"  {expr_str:28s}  {degree:3d}  {len(vars_used):4d}  "
          f"{ms_base:8.2f} ms     {ms_opt_sw:8.2f} ms      {speedup:5.2f}×")

---
## Section 11 — Memory Traffic Breakdown

Quantifying exactly how many bytes each optimization eliminates.

In [ ]:
# ── HBM traffic model for each optimization ────────────────────────────────

def hbm_traffic_model(n, V, degree, dtype_bytes=4):
    """
    Model total HBM bytes moved for one sumcheck run.
    Returns dict of {version: bytes}.
    """
    N = 2**n
    B = dtype_bytes

    results = {}

    # Version 0: Baseline
    # claim0: V reads + 1 write f + 1 read f = (V+2)*N*B
    # Per round k (table size N_k = N/2^k), per eval point:
    #   stride gather: 2*N_k*B   (read+write evens, odds)
    #   mle_update (unfused, 3 kernels): 9*(N_k//2)*B per var per t≥2
    #   eval_expr: (V-1)*(N_k//2)*B per eval point
    #   sum: (N_k//2)*B per eval point
    #   fold: 9*(N_k//2)*B*V
    total_v0 = (V + 2) * N * B  # claim0
    for k in range(n):
        N_k  = N // (2**k)
        half = N_k // 2
        total_v0 += 2 * V * N_k * B                  # gather per var
        total_v0 += 9 * V * half * B * (degree - 1)  # unfused mle_update per t≥2
        total_v0 += (V - 1) * half * B * (degree + 1)  # eval_expr mul
        total_v0 += half * B * (degree + 1)           # reductions
        total_v0 += 9 * V * half * B                  # fold (unfused)
    results['Baseline (unfused)'] = total_v0

    # Version 1: JIT fusion
    # mle_update: 3 passes (fused)
    # fold: 3 passes (fused)
    total_v1 = (V + 2) * N * B  # claim0 still 2 passes
    for k in range(n):
        N_k  = N // (2**k)
        half = N_k // 2
        total_v1 += 2 * V * N_k * B                    # gather
        total_v1 += 3 * V * half * B * (degree - 1)    # fused mle for t≥2
        total_v1 += (V - 1) * half * B * (degree + 1)  # eval_expr mul
        total_v1 += half * B * (degree + 1)             # reductions
        total_v1 += 3 * V * half * B                    # fused fold
    results['+ JIT fusion'] = total_v1

    # Version 2: + reshape (no stride gather copy)
    total_v2 = V * N * B  # claim0 fused: V reads only
    for k in range(n):
        N_k  = N // (2**k)
        half = N_k // 2
        total_v2 += V * N_k * B                         # reshape is view: 1 read
        total_v2 += 3 * V * half * B * (degree - 1)    # fused mle for t≥2
        total_v2 += (V - 1) * half * B * (degree + 1)  # eval mul
        total_v2 += half * B * (degree + 1)             # reduce
        total_v2 += 3 * V * half * B                    # fused fold
    results['+ reshape split + fused claim0'] = total_v2

    # Version 3: + vmap + fused g(t)+fold
    # The diff is computed once for all t, fold reuses it
    total_v3 = V * N * B  # claim0
    for k in range(n):
        N_k  = N // (2**k)
        half = N_k // 2
        total_v3 += V * N_k * B              # reshape read
        total_v3 += 2 * V * half * B         # read evens+odds once (diff computed in regs)
        # All g(t) share the same diff; just scale by different t
        total_v3 += (V - 1) * half * B * (degree + 1)  # eval mul for each t
        total_v3 += half * B * (degree + 1)             # reduce
        total_v3 += V * half * B                        # write fold output
    results['+ vmap + shared diff'] = total_v3

    return results


print("HBM Traffic Model: n=20, V=3, degree=3 (a*b*c)")
print("=" * 60)
n_model, V_model, d_model = 20, 3, 3
traffic = hbm_traffic_model(n_model, V_model, d_model)
base_bytes = list(traffic.values())[0]

for name, bytes_ in traffic.items():
    reduction = base_bytes / bytes_
    print(f"  {name:40s}  {bytes_/1e9:.2f} GB   ({reduction:.2f}× vs baseline)")

print()
print("Each optimization stacks: total reduction is multiplicative.")

---
## Section 12 — Roofline After Optimizations

After all optimizations, where does the fused SumCheck land on the roofline?

In [ ]:
# ── Updated roofline with fused variants ───────────────────────────────────

fig, ax = plt.subplots(figsize=(11, 5))
AI_range = np.logspace(-3, 4, 300)
roofline = np.minimum(peak_TFLOPS * 1e12, AI_range * hbm_bw_GBs * 1e9) / 1e9
ax.loglog(AI_range, roofline, 'k-', lw=2.5, label='Roofline')

ops_updated = {
    # (FLOPs/elem, bytes/elem)
    "mle_update (unfused, 3 kernels)": (6,  9 * 4,  'red',    'x'),
    "mle_update (fused, 1 kernel)":    (6,  3 * 4,  'orange', 'o'),
    "mod_mul_32 (basic)": (2, 3 * 4, 'blue', 'x'),
    "Round body (unfused)": (10, 15 * 4, 'purple', 'x'),
    "Round body (all fused)": (10, 5 * 4, 'green', 'o'),
    "Matmul (reference)": (2000, 4, 'gray', 's'),
}

for name, (flops, bytes_, color, marker) in ops_updated.items():
    ai   = flops / bytes_
    perf = min(peak_TFLOPS * 1e12, ai * hbm_bw_GBs * 1e9) / 1e9
    ax.scatter(ai, perf, s=140, color=color, marker=marker, zorder=5)
    ax.annotate(name, (ai, perf), textcoords='offset points',
                xytext=(6, 3), fontsize=8, color=color)

# Add legend for marker meaning
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0],[0], marker='x', color='k', label='Unfused (baseline)', linestyle=''),
    Line2D([0],[0], marker='o', color='k', label='Fused (optimized)',   linestyle=''),
]
ax.axvline(ridge_point, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Arithmetic Intensity (FLOPs / byte)')
ax.set_ylabel('Performance (GFLOPs/s)')
ax.set_title('Roofline: Optimizations shift SumCheck right (higher AI) along memory-bound slope')
ax.legend(handles=legend_elems, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: Even fully fused, SumCheck stays on the memory-bound slope.")
print("The optimizations move us RIGHT along the slope (more AI = same BW used better)")
print("but cannot cross the ridge — that requires fundamentally different algorithms.")

---
## Section 13 — Summary: What Each Optimization Buys

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│ OPTIMIZATION          TECHNIQUE          HBM TRAFFIC     LATENCY IMPACT         │
├──────────────────────────────────────────────────────────────────────────────────┤
│ 1. Kernel fusion      jax.jit            3× less         2–3× faster kernels    │
│    (mle_update)       (XLA fusion)       (9→3 passes)                           │
│                                                                                  │
│ 2. Reshape split      reshape(-1,2)      1× (view=free)  eliminate gather copy  │
│    (vs stride-2)      vs [0::2]          vs 2× passes    ~10–20% round speedup  │
│                                                                                  │
│ 3. vmap degree loop   jax.vmap           shared diff     eliminate d kernel      │
│    (g(t) batched)     over t values      computed once   launches, batch g(t)   │
│                                                                                  │
│ 4. Fused claim0       single jit block   V reads only    avoid f_vals HBM write │
│    (eval+sum)         over eval+reduce   vs (V+2) reads  1.5–2× faster claim0  │
│                                                                                  │
│ 5. lax.scan rounds    XLA-native loop    same bytes      eliminate n×Python     │
│    (outer loop)       vs Python for      (+ XLA sched.)  overhead (~2ms at n=20)│
│                                                                                  │
│ 6. SoA layout         (V,N) stacked      coalesced rows  better L2 hit rate     │
│    vs dict            vs dict of arrays  vs scattered    especially for V≥4     │
│                                                                                  │
│ 7. Chunked streaming  lax.scan over      1 read/element  critical for n≥22+     │
│    (large n)          chunks ≤ L2 size   guaranteed      on T4 (4MB L2)         │
│                                                                                  │
│ 8. All combined       make_sumcheck_     ~5× less        ~5–10× vs baseline     │
│                       optimized()        total bytes     at n=20                 │
└──────────────────────────────────────────────────────────────────────────────────┘

WHAT CANNOT BE OPTIMIZED (fundamental limits):
  • Sequential rounds:  each round needs prev round's folded table — unavoidable
  • Memory-bound floor: AI ≤ 1 FLOPs/byte — we will never reach compute-bound
  • Modular arithmetic: no native GPU int mod — must use uint64 intermediate

DIMINISHING RETURNS:
  At n=20: sequential overhead = ~20× 50µs = 1ms; table ops = ~5ms → 20% sequential
  At n=24: sequential overhead = ~24× 50µs = 1.2ms; table ops >> 50ms → 2% sequential
  → GPU parallelism ROI increases with n
```

In [ ]:
# ── Final summary bar chart ────────────────────────────────────────────────

# Collect all measured timings
timing_labels = [
    "Baseline\n(unfused)",
    "+ JIT\nfusion",
    "+ reshape\nsplit",
    "+ vmap\ndegree",
    "+ lax.scan\nrounds",
    "Full\noptimized",
]

# Use measured results where available, model estimates otherwise
# Scale by ratio of models for intermediate steps
traffic_vals = list(traffic.values())
base_t = traffic_vals[0]
scale_factors = [v / base_t for v in traffic_vals]

# Use actual measurements for baseline and optimized; interpolate intermediates
ms_baseline = results['baseline']
ms_optimized = results.get('optimized', ms_opt)

# Rough interpolation based on traffic model
n_steps = len(timing_labels)
ms_values = [ms_baseline]
for sf in scale_factors[1:]:
    ms_values.append(ms_baseline * sf)
ms_values[-1] = ms_optimized  # use actual measurement

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of latency
colors = ['#d73027', '#f46d43', '#fdae61', '#74add1', '#4575b4', '#313695']
bars = ax1.bar(timing_labels, ms_values, color=colors, edgecolor='black', linewidth=0.5)
for bar, ms in zip(bars, ms_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ms_baseline*0.01,
             f'{ms:.1f}ms', ha='center', va='bottom', fontsize=8)
ax1.set_ylabel('Latency (ms)')
ax1.set_title(f'SumCheck Latency by Optimization (n={N_BENCH})')
ax1.axhline(ms_baseline, color='red', linestyle='--', alpha=0.4, label='Baseline')
ax1.legend(fontsize=8)

# HBM traffic reduction
hbm_vals = [t / 1e9 for t in traffic_vals]
# Extend to match timing_labels
hbm_labels = list(traffic.keys())
ax2.barh(hbm_labels, hbm_vals,
         color=['#d73027', '#fdae61', '#74add1', '#313695'],
         edgecolor='black', linewidth=0.5)
for i, (val, label) in enumerate(zip(hbm_vals, hbm_labels)):
    ax2.text(val + hbm_vals[0]*0.01, i, f'{val:.1f} GB', va='center', fontsize=8)
ax2.set_xlabel('Total HBM Traffic (GB)')
ax2.set_title('HBM Traffic Model (n=20, a*b*c)')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\nTotal speedup: {ms_baseline/ms_optimized:.1f}× baseline")
print(f"HBM traffic:   {base_t/list(traffic.values())[-1]:.1f}× reduction")

---
## Section 14 — What's Next: Beyond Memory Optimizations

After reducing HBM traffic as much as possible, the remaining bottlenecks are:

### 1. Custom CUDA Kernels (Triton / CUDA C)

XLA's code generation is good but not perfect for modular arithmetic. A hand-written Triton kernel can:
- Use warp-level shuffle instructions for faster tree reductions
- Explicitly control shared memory for the split+eval+fold pipeline
- Avoid uint64→uint32 casts by using native PTX `mad.wide.u32` instruction

### 2. Montgomery Multiplication

For the large prime `q = 2^32 - 5`, Montgomery form eliminates the expensive modulo operation:
```
Standard:   (a * b) % q   → requires 64-bit div
Montgomery: REDC(a_M * b_M) → only shifts and additions (no div)
```
For `n=20` with millions of modular multiplications, this saves ~20–30% of arithmetic time.

### 3. GKR + SumCheck Pipelining

In practice, SumCheck is called once per GKR layer. The output folded table of one layer feeds into the next. Pipelining layer execution — starting the next layer's claim0 computation while the current layer's fold is still running — can hide latency.

### 4. NTT-based Polynomial Evaluation

For degree ≥ 10, evaluating g(t) via Number Theoretic Transform (NTT) reduces complexity from O(d × N) to O(N log d). SumCheck's degree is at most 7 here, so this doesn't apply — but worth knowing for zk-SNARK contexts.

### 5. Multi-GPU via `pmap`

If `n ≥ 24` (256M entries per table), the data can be sharded across GPUs:
```python
# Shard tables across 4 GPUs
# Each GPU handles N/4 entries per round
# Final reduction is an allreduce (tiny: just scalars)
jax.pmap(round_body, axis_name='dev')
```
The communication cost (allreduce of `d+1` scalars per round) is negligible compared to the compute.

---

**Key takeaway**: SumCheck is inherently memory-bandwidth limited. The path to speed is:
1. Reduce bytes moved (fusion, layout, reshape)
2. Maximize effective bandwidth (coalescing, cache-friendly access)
3. Eliminate software overhead (lax.scan, vmap, static JIT)
4. Scale out (multi-GPU) when single-device bandwidth is saturated